# Model Evaluation

Evaluasi model KNN (TF-IDF + Cosine Similarity) untuk book recommendation.

### Metrik:
1. **Precision@5** - Fraction of top-5 rekomendasi yang share genre dengan input (PRD: >80%)
2. **Recall@5** - Fraction of query's genres yang ter-cover di top-5 (PRD: >75%)
3. **Hit Rate@5** - Persentase buku yang punya minimal 1 rekomendasi relevan di top-5
4. **NDCG@5** - Ranking quality
5. **MRR** - Mean Reciprocal Rank
6. **Cosine Similarity** & **Diversity** - Qualitative metrics

### Ground Truth:
Genre-based: buku dianggap "relevan" jika share minimal 1 genre dengan query book.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import joblib
from sklearn.metrics.pairwise import cosine_similarity

GLOBAL_SUFFIX = "Young Adult"

def extract_book_genres(genres_str):
    if pd.isna(genres_str):
        return []
    parts = genres_str.split(",")
    last_global_idx = -1
    for i, p in enumerate(parts):
        if p.strip() == GLOBAL_SUFFIX:
            last_global_idx = i
    return list(set(g.strip() for g in parts[last_global_idx + 1:] if g.strip()))

books = pd.read_csv("../data/processed/books_for_ml.csv")
books = books.drop_duplicates(subset=["title"]).reset_index(drop=True)

raw = pd.read_csv("../data/raw/books.csv")
raw["book_genres"] = raw["genres"].apply(extract_book_genres)
genre_map = raw.set_index("bookId")["book_genres"].to_dict()
books["genres_clean"] = books["bookId"].map(genre_map).apply(
    lambda x: x if isinstance(x, list) else []
)
books["num_genres"] = books["genres_clean"].apply(len)

model = joblib.load("../ml/models/knn_model.pkl")
tfidf_matrix = joblib.load("../ml/models/tfidf_matrix.pkl")
title_to_index = joblib.load("../ml/models/title_to_index.pkl")

print(f"Total buku: {len(books)}")
print(f"Buku dengan genres: {(books['num_genres'] > 0).sum()}")
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

---
## 1. Precision@5 & Recall@5 (PRD Metrics)

In [ ]:
def compute_metrics(rec_indices, query_genres, books_df, k=5):
    # Precision@5: fraction of top-k that share any genre
    p_hits = 0
    for ri in rec_indices[:k]:
        if set(books_df.iloc[ri]["genres_clean"]) & query_genres:
            p_hits += 1
    precision = p_hits / k

    # Recall@5: fraction of query's genres covered by top-5
    covered_genres = set()
    for ri in rec_indices[:k]:
        covered_genres |= set(books_df.iloc[ri]["genres_clean"]) & query_genres
    recall = len(covered_genres) / len(query_genres) if query_genres else 0.0

    # Hit Rate
    hit = 1.0 if p_hits > 0 else 0.0

    # NDCG@5
    dcg = 0.0
    for i, ri in enumerate(rec_indices[:k]):
        rel = 1.0 if set(books_df.iloc[ri]["genres_clean"]) & query_genres else 0.0
        dcg += rel / np.log2(i + 2)
    ideal_dcg = sum(1.0 / np.log2(i + 2) for i in range(min(k, len(query_genres))))
    ndcg = dcg / ideal_dcg if ideal_dcg > 0 else 0.0

    # MRR
    mrr = 0.0
    for i, ri in enumerate(rec_indices):
        if set(books_df.iloc[ri]["genres_clean"]) & query_genres:
            mrr = 1.0 / (i + 1)
            break

    return {"precision@5": precision, "recall@5": recall, "hit_rate": hit, "ndcg@5": ndcg, "mrr": mrr}

In [ ]:
eval_books = books[books["num_genres"] >= 2].copy()
np.random.seed(42)
sample_indices = np.random.choice(len(eval_books), size=min(500, len(eval_books)), replace=False)
sample = eval_books.iloc[sample_indices]

all_metrics = []
for idx, row in sample.iterrows():
    title = row["title"]
    query_genres = set(row["genres_clean"])
    if title not in title_to_index:
        continue
    book_idx = title_to_index[title]
    distances, indices = model.kneighbors(tfidf_matrix[book_idx], n_neighbors=21)
    rec_indices = indices[0][1:]
    m = compute_metrics(rec_indices, query_genres, books, k=5)
    m["title"] = title
    all_metrics.append(m)

df = pd.DataFrame(all_metrics)
print(f"Books evaluated: {len(df)}")
print()
for metric in ["precision@5", "recall@5", "hit_rate", "ndcg@5", "mrr"]:
    vals = df[metric]
    print(f"{metric.upper():15s}: mean={vals.mean():.4f} ({vals.mean()*100:.1f}%)  median={vals.median():.4f}")

print()
print("=" * 60)
print("PRD TARGET CHECK")
print("=" * 60)
p5 = df["precision@5"].mean()
r5 = df["recall@5"].mean()
print(f"Precision@5 > 80%: {p5*100:.1f}% -> {'PASS' if p5 >= 0.80 else 'FAIL'}")
print(f"Recall@5 > 75%:    {r5*100:.1f}% -> {'PASS' if r5 >= 0.75 else 'FAIL'}")

---
## 2. Manual Sampling

Coba beberapa judul buku dan cek apakah rekomendasinya relevan.

In [ ]:
def get_recommendations(book_title, top_n=5):
    if book_title not in title_to_index:
        return None
    idx = title_to_index[book_title]
    distances, indices = model.kneighbors(tfidf_matrix[idx], n_neighbors=top_n + 1)
    rec_indices = indices[0][1:]
    rec_distances = distances[0][1:]
    rec_books = books.iloc[rec_indices][["title", "author"]].reset_index(drop=True)
    rec_books["cosine_similarity"] = 1 - rec_distances
    rec_books["genres"] = [books.iloc[ri]["genres_clean"] for ri in rec_indices]
    return rec_books

sample_books = ["1984", "Harry Potter and the Sorcerer's Stone", "The Hobbit",
                "To Kill a Mockingbird", "The Great Gatsby", "Linear Algebra Done Right"]

for title in sample_books:
    recs = get_recommendations(title)
    print(f"\n{'='*60}")
    print(f"Input: {title}")
    if recs is None:
        print("-> BUKU TIDAK DITEMUKAN")
        continue
    for i, row in recs.iterrows():
        print(f"  {i+1}. {row['title'][:55]:55s} sim={row['cosine_similarity']:.3f} genres={row['genres'][:3]}")

---
## 3. Ringkasan

In [ ]:
print("=" * 60)
print("RINGKASAN EVALUASI MODEL")
print("=" * 60)
print(f"Model: KNN (TF-IDF + genres, cosine, brute)")
print(f"Total buku: {len(books)}")
print(f"Buku dievaluasi: {len(df)}")
print()
print("PRD Metrics:")
print(f"  Precision@5: {df['precision@5'].mean()*100:.1f}% (target: >80%)")
print(f"  Recall@5:    {df['recall@5'].mean()*100:.1f}% (target: >75%)")
print()
print("Additional Metrics:")
print(f"  Hit Rate:    {df['hit_rate'].mean()*100:.1f}%")
print(f"  NDCG@5:      {df['ndcg@5'].mean()*100:.1f}%")
print(f"  MRR:         {df['mrr'].mean()*100:.1f}%")